# Does the text of an agent skill predict its replication on GitHub?

[![Open In Colab](https://colab.research.google.com/assets/colab/badge.svg)](https://colab.research.google.com/github/podorize/skillreuse/blob/main/notebooks/reproduce.ipynb)

Companion notebook to the paper of Pedro de Paula Odorize and Rafael Borges Allgayer for CMP263 (Machine Learning,
UFRGS). Run it while you read the [paper](https://github.com/podorize/skillreuse/blob/main/paper/latex/main.pdf):
each section below has the number and name of a section of the paper, and its code produces the numbers, tables and
figures of that section.

The notebook does the whole analysis: from the dated copies to the study table, the models, the evaluation and the
interpretation. The collection is not done here. Choosing the sample from GitSkills (6 GB) and dating every copy
through the GitHub API (about seven hours) were done by the code of the
[repository](https://github.com/podorize/skillreuse), and their outputs are downloaded from it in the first cell. To
run everything from scratch, follow the README of the repository.

Running all cells takes about 30 to 40 minutes on Colab; sections 4.2 and 4.5 are the slowest.

## 0. Setup

The first cell downloads from GitHub the files that the analysis reads. When the notebook runs inside a copy of the
repository, the files are already there and nothing is downloaded.

In [ ]:
import os
import sys
import urllib.request
from pathlib import Path

if Path.cwd().name == "notebooks" and Path("../config/config.yaml").exists():
    os.chdir("..")  # running inside the repository: use its files

GITHUB = "https://raw.githubusercontent.com/podorize/skillreuse/main/"


def get(path):
    """Local copy of a file of the repository, downloaded from GitHub the first time."""
    local = Path(path)
    if not local.exists():
        local.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(GITHUB + path, local)
    return local


FILES = [
    "uv.lock",                                              # exact versions of the libraries
    "config/config.yaml",                                   # settings of the study
    "results/phase1/manifest.json",                         # counts of the collection
    "data/derived/collection/sample_occurrences.parquet",   # every copy of the sampled skills
    "data/derived/collection/dates.parquet",                # arrival of each copy (GitHub API)
    "data/derived/collection/text_attributes.parquet",      # attributes of the text of each skill
    "data/derived/collection/repositories.parquet",         # language, license, stars of each repository
    "data/derived/collection/origin_stars.parquet",         # stars with their dates (GitHub API)
]
for path in FILES:
    print(f"{get(path).stat().st_size / 1e6:6.1f} MB  {path}")

The study used fixed versions of the libraries (`uv.lock`). This cell installs them when they differ, as on Colab. After
an installation, restart the session (Runtime > Restart session) and run all cells again.

In [ ]:
import importlib.metadata
import subprocess
import tomllib

locked = {p["name"]: p["version"] for p in tomllib.loads(Path("uv.lock").read_text())["package"]}
PACKAGES = ["numpy", "pandas", "pyarrow", "scipy", "scikit-learn", "matplotlib", "shap", "pyyaml"]


def installed(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


to_install = [f"{p}=={locked[p]}" for p in PACKAGES if installed(p) != locked[p]]
if to_install:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *to_install], check=True)
    raise SystemExit("Installed " + ", ".join(to_install) + ". Restart the session and run all cells again.")
print("All libraries have the versions used in the study.")

The settings of the study (seeds, dates, limits, lists of attributes and algorithms) are in `config/config.yaml`, the
same file that the code of the repository reads.

In [ ]:
import hashlib
import json
import re
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from scipy import stats
from scipy.stats import loguniform, randint
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, brier_score_loss, confusion_matrix, log_loss,
                             precision_recall_curve, roc_auc_score, roc_curve)
from sklearn.model_selection import RandomizedSearchCV, StratifiedGroupKFold
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore", category=ConvergenceWarning)  # some models stop before converging
pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 40)

CONFIG = yaml.safe_load(Path("config/config.yaml").read_text())
SNAPSHOT = pd.Timestamp(CONFIG["dating"]["snapshot"], tz="UTC")  # start of the GitSkills crawl
SEED = CONFIG["models"]["seed"]
FAMILIES = CONFIG["attributes"]["families"]                      # the 27 text attributes, by family
TEXT = [column for family in FAMILIES.values() for column in family]
ORIGIN = CONFIG["attributes"]["origin"]                          # attributes of the origin repository
CATEGORICAL = set(CONFIG["attributes"]["categorical"])
POPULARITY = CONFIG["attributes"]["popularity"]
ORIGIN_POP = [c for c in ORIGIN if c != "log_stars_at_t0"] + POPULARITY
print(f"{len(TEXT)} text attributes in {len(FAMILIES)} families; snapshot {SNAPSHOT.date()}; seed {SEED}")

---
## 3.1 Data

GitSkills lists every `SKILL.md` file found on public GitHub repositories. Grouping the files by the hash of their
content gives the distinct skills. A repository with 100 or more distinct skills is an aggregator, and the population
is every skill with at least one copy outside aggregators. These counts and the sample were computed by the collection
code on the 6 GB of GitSkills ([`collect.py`](https://github.com/podorize/skillreuse/blob/main/src/skillreuse/collect.py),
function `build_sample`); its manifest records them.

In [ ]:
counts = json.loads(Path("results/phase1/manifest.json").read_text())["counts"]
labels = {
    "gitskills_files": "SKILL.md files in GitSkills",
    "gitskills_repositories": "repositories in GitSkills",
    "gitskills_contents": "distinct skills (distinct contents)",
    "aggregators": "aggregators (repositories with 100 or more skills)",
    "largest_repository_skills": "skills in the largest aggregator",
    "population_occurrences": "copies of the population skills",
    "population_contents": "population: skills with a copy outside aggregators",
    "sample_contents": "skills in the sample",
    "occurrences": "copies of the sampled skills",
    "occurrences_outside_aggregators": "  outside aggregators",
    "occurrences_in_aggregators": "  in aggregators",
}
pd.Series({labels[key]: f"{value:,}" for key, value in counts.items()}, name="count").to_frame()

The sample itself is published: one row per copy of a sampled skill, with the number of distinct skills of its
repository (`repo_skills`). Counting its rows gives the last four numbers above.

In [ ]:
occ = pd.read_parquet("data/derived/collection/sample_occurrences.parquet")
print(f"skills: {occ.content_hash.nunique():,}")
print(f"copies: {len(occ):,} ({(~occ.aggregator).sum():,} outside aggregators, {occ.aggregator.sum():,} in aggregators)")
occ.head(3)

---
## 3.2 Measuring whether a skill was copied

For every copy, the collection asked the GitHub API for the history of the file and kept the first commit in which the
file has exactly the content of the skill. The arrival of a copy is that date or, if later, the creation of the
repository (a repository created by cloning another one inherits its old history). A copy that could not be dated has
no arrival. The dates are published:

In [ ]:
dates = pd.read_parquet("data/derived/collection/dates.parquet")
copies = occ.merge(dates, on=["content_hash", "repo_full_name", "path"], how="left")
outside = copies[~copies.aggregator]
# the status of each copy is stored in Portuguese
STATUS = {"datada": "dated", "repositorio_inacessivel": "repository gone", "arquivo_ausente": "file gone",
          "hash_nao_encontrado": "hash not found", "falha_de_coleta": "collection failure",
          "data_posterior_ao_retrato": "date after the snapshot"}
print(f"dated copies outside aggregators: {(outside.status == 'datada').sum():,} of {len(outside):,}")
outside.status.map(STATUS).value_counts().to_frame("copies")

With the arrivals, each skill gets its first appearance (T0, the earliest arrival outside aggregators), its
origin (the copy that arrived first) and its author (the owner of the origin repository). The time to the first
copy is counted from T0 to the first copy by another owner; copies by the author do not count.

In [ ]:
def first_appearance(copies, limit):
    """T0, origin, author and days to the first copy by another owner, for each skill.

    Copies in repositories with `limit` or more skills (aggregators) are left out;
    limit=None keeps every copy.
    """
    if limit is not None:
        copies = copies[copies["repo_skills"] < limit]
    dated = copies[copies["arrival"].notna()].sort_values(
        ["content_hash", "arrival", "repo_full_name", "path"])

    # the first dated copy of each skill is its origin; ties go to the first repository and path
    origin = dated.drop_duplicates("content_hash", keep="first")
    skills = origin[["content_hash", "arrival", "repo_full_name", "path", "owner", "repo_created_at"]]
    skills = skills.rename(columns={"arrival": "t0", "repo_full_name": "origin_repo",
                                    "path": "origin_path", "owner": "author",
                                    "repo_created_at": "origin_repo_created_at"})

    # days from T0 to the first copy by an owner other than the author
    others = dated.merge(skills[["content_hash", "author", "t0"]], on="content_hash")
    others = others[others["owner"] != others["author"]]
    first_other_copy = others.groupby("content_hash")["arrival"].min()
    t0 = others.groupby("content_hash")["t0"].first()
    days = (first_other_copy - t0).dt.total_seconds() / 86400
    skills["first_copy_lag_days"] = skills["content_hash"].map(days)

    every_copy_dated = copies.groupby("content_hash")["arrival"].apply(lambda a: bool(a.notna().all()))
    skills["all_copies_dated"] = skills["content_hash"].map(every_copy_dated)
    return skills

The example of the paper: Ana publishes a skill on 2 March, Bruno copies it on 12 March, Ana copies it into a second
project of hers, and a mirror with 5,000 skills collects it in April.

In [ ]:
example = pd.DataFrame({
    "content_hash": "skill",
    "repo_full_name": ["ana/project", "bruno/tools", "ana/other-project", "mirror/all-skills"],
    "path": "SKILL.md",
    "owner": ["ana", "bruno", "ana", "mirror"],
    "repo_skills": [1, 3, 1, 5000],
    "arrival": pd.to_datetime(["2026-03-02", "2026-03-12", "2026-03-20", "2026-04-10"], utc=True),
})
example["repo_created_at"] = example["arrival"]
first_appearance(example, limit=100)[["t0", "author", "first_copy_lag_days"]]

Ana is the author and Bruno's copy counts, 10 days after T0; her second project and the mirror do not. The same function
on every copy of the sample:

In [ ]:
skills = first_appearance(copies, limit=CONFIG["sample"]["aggregator_min_skills"])
print(f"skills with at least one dated copy outside aggregators: {len(skills):,}")
print(f"copied at some point by another developer: {skills.first_copy_lag_days.notna().sum():,}")

---
## 3.3 Attributes

The model reads only the `SKILL.md`. Each of its attributes is a count or a ratio computed with a regular expression,
listed in `config/config.yaml`:

In [ ]:
PATTERNS = CONFIG["text_patterns"]
pd.Series(PATTERNS, name="regular expression").to_frame()

The function below computes the attributes of one text. The collection code ran it on the text of every sampled skill
(the texts stay in GitSkills, under the license of each repository); here it runs on an example.

In [ ]:
def text_attributes(text, name, description):
    """The attributes of the text and of the front matter of one SKILL.md."""
    def count(pattern, flags=re.MULTILINE):
        return len(re.findall(PATTERNS[pattern], text, flags=flags))

    words = re.findall(PATTERNS["word"], text.lower())
    lines = text.splitlines()
    headings = re.findall(PATTERNS["heading"], text, flags=re.MULTILINE)  # the '#' marks
    sentences = count("sentence", flags=0) or (1 if words else 0)  # no final mark: one sentence
    return {
        "text_char_count": len(text),
        "text_line_count": len(lines),
        "text_nonempty_line_count": sum(bool(line.strip()) for line in lines),
        "text_word_count": len(words),
        "text_unique_word_count": len(set(words)),
        "text_lexical_diversity": len(set(words)) / len(words) if words else 0.0,
        "text_average_word_length": sum(map(len, words)) / len(words) if words else 0.0,
        "text_sentence_count": sentences,
        "text_average_sentence_words": len(words) / sentences if sentences else 0.0,
        "markdown_heading_count": len(headings),
        "markdown_max_heading_depth": max(map(len, headings), default=0),
        "markdown_unordered_list_items": count("unordered_list"),
        "markdown_ordered_list_items": count("ordered_list"),
        "markdown_fenced_code_blocks": count("fence") // 2,  # two fences make one block
        "markdown_inline_code_count": count("inline_code", flags=0),
        "markdown_link_count": count("markdown_link", flags=0),
        "markdown_url_count": count("url", flags=0),
        "markdown_blockquote_count": count("blockquote"),
        "markdown_checkbox_count": count("checkbox"),
        "markdown_table_row_count": count("table_row"),
        "markdown_shell_prompt_count": count("shell_prompt"),
        "frontmatter_name_char_count": len(name or ""),
        "frontmatter_description_char_count": len(description or ""),
        "frontmatter_name_missing": int(name is None),
        "frontmatter_description_missing": int(description is None),
    }

In [ ]:
skill_text = '''---
name: release-notes
description: Write the release notes of a version from its merged pull requests.
---
# Release notes

Read the merged pull requests since the last tag. Group them by type, and write one line per change.

## Steps
1. List the pull requests with `gh pr list --state merged`.
2. Group them into features, fixes and other changes.
- Keep each line short.

```bash
$ gh pr list --state merged --limit 100
```

See the [style guide](https://example.com/style).
'''
pd.Series(text_attributes(skill_text, "release-notes",
                          "Write the release notes of a version from its merged pull requests."),
          name="value").to_frame()

GitSkills gives two more attributes, whether the front matter is valid and the length of the body. The attributes of
all sampled skills:

In [ ]:
attributes = pd.read_parquet("data/derived/collection/text_attributes.parquet")
print(f"{len(attributes):,} skills")
attributes[TEXT].describe().T[["mean", "50%", "max"]].round(2)

The second group describes the origin repository, for the control model: its age at T0, language, license, whether
it is a fork, its popularity, and the folders in the path of the file. The stars at T0 come from the GitHub API, which
no longer reports when each star was given, so this column ends up constant; the control uses the stars and forks of
the repository at the snapshot instead.

In [ ]:
repos = pd.read_parquet("data/derived/collection/repositories.parquet")
star_dates = pd.read_parquet("data/derived/collection/origin_stars.parquet")
stars_known = dict(zip(star_dates.repo_full_name, star_dates.starred_at))
stars_ok = set(star_dates.loc[star_dates.status == "ok", "repo_full_name"])
FOLDERS = CONFIG["cohort"]["folder_patterns"]


def origin_attributes(skills):
    """Attributes of the origin repository of each skill."""
    out = skills[["content_hash", "origin_repo", "origin_path", "t0", "origin_repo_created_at"]]
    out = out.merge(repos.rename(columns={"repo_full_name": "origin_repo"}), on="origin_repo", how="left")
    # negative when T0 is before the creation of the repository (an inherited history)
    out["repo_age_days_at_t0"] = (out["t0"] - out["origin_repo_created_at"]).dt.total_seconds() / 86400

    def stars_before(repo, t0):
        if repo not in stars_ok:
            return np.nan
        before = sum(pd.Timestamp(date) < t0 for date in stars_known[repo])
        return float(min(before, CONFIG["dating"]["star_cap"]))

    out["log_stars_at_t0"] = np.log1p([stars_before(r, t) for r, t in zip(out["origin_repo"], out["t0"])])
    out["log_stars_snapshot"] = np.log1p(out["stars"].astype(float))
    out["log_forks_snapshot"] = np.log1p(out["forks"].astype(float))
    path = out["origin_path"].fillna("")
    out["path_depth"] = path.str.count("/").astype(float)
    out["path_is_root"] = path.str.fullmatch(r"(?i)skill\.md").astype(float)
    for name, pattern in FOLDERS.items():
        out[name] = path.str.contains(pattern, case=False, regex=True).astype(float)
    out = out.rename(columns={"language": "repo_language", "license": "repo_license",
                              "is_fork": "repo_is_fork"})
    out["repo_is_fork"] = out["repo_is_fork"].astype(float)
    return out[["content_hash", *ORIGIN, *POPULARITY]]


pd.Series(FOLDERS, name="folder pattern").to_frame()

---
## 3.4 Training and test data

Each author keeps at most 50 skills, the first ones in hash order. Then the authors are split: in each size band (1,
2–3, 4–10 and 11–50 skills), the authors are ordered by a hash of their name and the first 30% form the holdout;
the others form the development set.

In [ ]:
def split_authors(skills):
    """True for the authors of the holdout."""
    per_author = skills.groupby("author").size()
    authors = pd.DataFrame({"author": per_author.index, "skills": per_author.to_numpy()})
    authors["band"] = pd.cut(authors["skills"], [0, 1, 3, 10, 50, np.inf],
                             labels=["1", "2-3", "4-10", "11-50", "51+"]).astype(str)
    seed = CONFIG["cohort"]["split_seed"]
    authors["hash"] = [hashlib.md5(f"{a}:{seed}".encode()).hexdigest() for a in authors["author"]]
    authors = authors.sort_values(["band", "hash", "author"])
    authors["position"] = authors.groupby("band").cumcount() + 1
    authors["band_size"] = authors.groupby("band")["author"].transform("size")
    authors["holdout"] = authors["position"] <= (CONFIG["cohort"]["holdout_share"] * authors["band_size"]).round()
    return authors.set_index("author")["holdout"]


skills = skills.sort_values(["author", "content_hash"])
skills = skills[skills.groupby("author").cumcount() < CONFIG["cohort"]["cap_per_author"]]
holdout_author = split_authors(skills)

The study table puts together, for each skill, its timing, its split, its text attributes and its origin attributes.

In [ ]:
table = (skills.merge(attributes, on="content_hash", validate="1:1")
               .merge(origin_attributes(skills), on="content_hash", validate="1:1")
               .rename(columns={"author": "owner"})
               .sort_values("content_hash").reset_index(drop=True))
table["split"] = np.where(table["owner"].map(holdout_author), "holdout", "dev")

for side in ("dev", "holdout"):
    part = table[table.split == side]
    print(f"{side:8s} {len(part):,} skills from {part.owner.nunique():,} authors")
dev_authors, holdout_authors = set(table.owner[table.split == "dev"]), set(table.owner[table.split == "holdout"])
print("authors on both sides:", len(dev_authors & holdout_authors))
print("most skills of one author:", table.groupby("owner").size().max())

---
## 3.5 Exploratory analysis and preprocessing

Only the development set is used here. The window W is chosen by a rule fixed in advance: the shortest of 30, 60 and 90
days that covers at least 80% of the first copies, among skills observed for at least 90 days.

In [ ]:
dev_all = table[table.split == "dev"]
observed = dev_all[dev_all.t0 <= SNAPSHOT - pd.Timedelta(days=90)]
lags = observed.first_copy_lag_days[observed.first_copy_lag_days <= 90]
coverage = {w: (lags <= w).mean() for w in CONFIG["models"]["windows"]}
W = next((w for w in CONFIG["models"]["windows"] if coverage[w] >= 0.80), 90)
print("share of first copies within W days:", {w: round(share, 3) for w, share in coverage.items()})
print("chosen window W:", W, "days")

A skill is eligible if its whole window ends before the snapshot, and copied (the target) if another developer
copied it within W days. This function returns the eligible skills of one side, with the target:

In [ ]:
def eligible(side, w, data=None, suffix=""):
    """Eligible skills of one side of the split, with the target `replicated` (1 = copied)."""
    data = table if data is None else data
    frame = data[data["split"] == side].copy()
    t0 = pd.to_datetime(frame[f"t0{suffix}"], utc=True)
    frame["replicated"] = (frame[f"first_copy_lag_days{suffix}"] <= w).astype(int)
    frame["t0_used"] = t0
    frame = frame[t0.notna() & (t0 <= SNAPSHOT - pd.Timedelta(days=w))]
    return frame.sort_values("content_hash").reset_index(drop=True)


dev, hold = eligible("dev", W), eligible("holdout", W)
for name, frame in (("development", dev), ("holdout", hold)):
    print(f"{name:12s} {len(frame):,} eligible skills from {frame.owner.nunique():,} authors, "
          f"{frame.replicated.mean():.1%} copied")

Four rules turn the exploratory findings into preprocessing (Table 2): constant columns are removed; numeric columns
with skewness above 2 get log(1 + x) in models sensitive to scale; of two columns with Spearman correlation above 0.95,
only the first stays in those models; impossible values (a negative age) become missing.

In [ ]:
def preprocessing_rules(dev):
    columns = TEXT + ORIGIN
    numeric = [c for c in columns if c not in CATEGORICAL]
    constant = [c for c in columns if dev[c].nunique(dropna=True) <= 1]
    candidates = [c for c in numeric if c not in constant]
    skewed = [c for c in candidates if dev[c].dropna().min() >= 0 and dev[c].nunique() > 2
              and dev[c].dropna().astype(float).skew() > 2]
    correlation = dev[candidates].astype(float).corr("spearman").abs()
    pairs = [(a, b) for i, a in enumerate(candidates) for b in candidates[i + 1:] if correlation.loc[a, b] > 0.95]
    kept, dropped = [], []
    for c in candidates:  # keep a column unless it correlates above 0.95 with one already kept
        (dropped if any(correlation.loc[c, k] > 0.95 for k in kept) else kept).append(c)
    return {"constant": constant, "log": skewed, "pairs": pairs, "drop": dropped}


rules = preprocessing_rules(dev.assign(repo_age_days_at_t0=dev.repo_age_days_at_t0.where(dev.repo_age_days_at_t0 >= 0)))
print("constant columns:", rules["constant"])
print("columns with log(1 + x):", len(rules["log"]))
print("pairs with Spearman above 0.95:", len(rules["pairs"]))
print("columns removed for collinearity:", rules["drop"])

The rules become a scikit-learn pipeline. Everything is fitted inside each training fold, so nothing from the
validation data leaks into training. Trees skip the log, the scaling and the removal of correlated columns.

In [ ]:
TREES = set(CONFIG["models"]["tree_algorithms"])
VIEWS = {"text": TEXT, "size": FAMILIES["Size"], "origin_pop": ORIGIN_POP, "origin_pop+text": ORIGIN_POP + TEXT}


def estimator(algo, threads=-1):
    """A new model with the default hyperparameters of scikit-learn."""
    return {"logistic": LogisticRegression(max_iter=2000),
            "naive_bayes": GaussianNB(),
            "knn": KNeighborsClassifier(n_jobs=threads),
            "decision_tree": DecisionTreeClassifier(random_state=SEED),
            "random_forest": RandomForestClassifier(random_state=SEED, n_jobs=threads),
            "hist_gb": HistGradientBoostingClassifier(random_state=SEED),
            "mlp": MLPClassifier(random_state=SEED)}[algo]


def log1p_clip(x):
    return np.log1p(np.clip(np.asarray(x, dtype=float), 0, None))


def dense(x):
    return x.toarray() if hasattr(x, "toarray") else np.asarray(x)


def pipeline(algo, view, params=None, threads=-1):
    """Preprocessing and model for one algorithm and one set of columns."""
    tree = algo in TREES
    cols = [c for c in VIEWS[view] if c not in rules["constant"] and (tree or c not in rules["drop"])]
    categorical = [c for c in cols if c in CATEGORICAL]
    numeric = [c for c in cols if c not in CATEGORICAL]
    logged = [] if tree else [c for c in numeric if c in rules["log"]]
    plain = [c for c in numeric if c not in logged]
    blocks = []
    if plain:
        steps = [("impute", SimpleImputer(strategy="median", add_indicator=True))]
        blocks.append(("num", Pipeline(steps if tree else steps + [("scale", StandardScaler())]), plain))
    if logged:
        blocks.append(("num_log", Pipeline([("impute", SimpleImputer(strategy="median", add_indicator=True)),
                                            ("log", FunctionTransformer(log1p_clip, feature_names_out="one-to-one")),
                                            ("scale", StandardScaler())]), logged))
    if categorical:
        blocks.append(("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
                                        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=10))]),
                       categorical))
    steps = [("pre", ColumnTransformer(blocks, remainder="drop"))]
    if algo != "logistic":
        steps.append(("dense", FunctionTransformer(dense, accept_sparse=True)))
    model = Pipeline(steps + [("model", estimator(algo, threads))])
    if params:
        model.set_params(**{f"model__{k}": v for k, v in params.items()})
    return model, cols


model, cols = pipeline("logistic", "text")
print(len(cols), "text columns in the logistic regression")
model

---
## 3.6 Model development

Every algorithm is trained and validated on the same partitions of the development set: 3 folds, repeated 3 times
with different seeds, grouped by author and stratified by the target.

In [ ]:
def folds(frame, repeats=3):
    """(repetition, fold, training rows, validation rows) of the repeated, author-grouped folds."""
    authors, y = frame["owner"].astype(str).to_numpy(), frame["replicated"].to_numpy()
    for repetition in range(1, repeats + 1):
        splitter = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED + repetition - 1)
        for fold, (train, valid) in enumerate(splitter.split(frame, y=y, groups=authors), 1):
            yield repetition, fold, train, valid

The three best algorithms of the spot-check are tuned by a random search of ten configurations (Table 3), scored on
three inner folds of the training part.

In [ ]:
def space(values):
    """One search space of the settings: a list of values, or {loguniform | randint: [low, high]}."""
    if isinstance(values, dict):
        (name, (low, high)), = values.items()
        return {"loguniform": loguniform, "randint": randint}[name](low, high)
    return [tuple(v) if isinstance(v, list) else v for v in values]


SPACES = {algo: {name: space(v) for name, v in params.items()}
          for algo, params in CONFIG["models"]["search_spaces"].items()}


def search(algo, train, seed):
    """Random search of the hyperparameters of one algorithm on the training part."""
    model, cols = pipeline(algo, "text", threads=1)
    inner = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=seed)
    inner_folds = list(inner.split(train, y=train["replicated"], groups=train["owner"].astype(str)))
    grid = {f"model__{k}": v for k, v in SPACES[algo].items()}
    rs = RandomizedSearchCV(model, grid, n_iter=10, cv=inner_folds, scoring="roc_auc", n_jobs=4,
                            random_state=SEED, refit=True, error_score="raise")
    return rs.fit(train[cols], train["replicated"].to_numpy())


for algo in ("hist_gb", "logistic", "mlp"):
    print(algo, CONFIG["models"]["search_spaces"][algo])

Two configurations are compared fold by fold. The folds share training data, so the paired t test uses the
Nadeau–Bengio correction of the variance, and Holm's method adjusts the p-values for several comparisons.

In [ ]:
def nadeau_bengio(diff, ratio):
    """Paired t test on per-fold differences; ratio = validation size / training size."""
    n, mean = len(diff), float(np.mean(diff))
    std_error = np.sqrt((1 / n + ratio) * np.var(diff, ddof=1))
    t_critical = stats.t.ppf(0.975, n - 1)
    p = float(2 * stats.t.sf(abs(mean / std_error), n - 1)) if std_error > 0 else float(mean == 0)
    return {"mean_diff": mean, "ci95_low": mean - t_critical * std_error,
            "ci95_high": mean + t_critical * std_error, "p": p}


def holm(p_values):
    """Holm's adjustment: the k-th smallest p-value times (m - k + 1), never decreasing."""
    adjusted, largest = np.empty(len(p_values)), 0.0
    for rank, i in enumerate(np.argsort(p_values)):
        largest = max(largest, (len(p_values) - rank) * p_values[i])
        adjusted[i] = min(1.0, largest)
    return adjusted.tolist()


def compare_to_best(results):
    """Test the configuration with the highest mean AUC against each of the others."""
    auc = results.pivot_table(index=["repetition", "fold"], columns="config", values="auc")
    best = auc.mean().idxmax()
    ratio = (results["valid_rows"] / results["train_rows"]).mean()
    tests = pd.DataFrame([{"best": best, "other": other, **nadeau_bengio((auc[best] - auc[other]).to_numpy(), ratio)}
                          for other in auc.columns if other != best])
    tests["p_holm"] = holm(tests["p"].tolist())
    return tests

---
## 3.7 Evaluation

The main metric is the AUC. The 95% intervals come from bootstrap samples that draw whole authors, since the skills
of one author are not independent.

In [ ]:
N_BOOT = CONFIG["evaluation"]["n_boot"]


def metrics(y, p):
    """AUC, PR AUC, Brier score and log loss. The probabilities are clipped to [1e-6, 1 - 1e-6] first, as in every
    metric of the study, so that the log loss is never infinite; this also ties the probabilities of exactly 0 or 1
    that Naive Bayes often gives."""
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return {"roc_auc": roc_auc_score(y, p), "pr_auc": average_precision_score(y, p),
            "brier": brier_score_loss(y, p), "log_loss": log_loss(y, p, labels=[0, 1])}


def author_bootstrap(y, p, owners, n_boot=N_BOOT):
    """AUC in each bootstrap sample; each skill is weighted by the times its author is drawn."""
    authors, author_of_row = np.unique(owners, return_inverse=True)
    rng = np.random.default_rng(SEED)
    aucs = []
    for _ in range(n_boot):
        drawn = np.bincount(rng.integers(0, len(authors), len(authors)), minlength=len(authors))
        aucs.append(roc_auc_score(y, p, sample_weight=drawn[author_of_row].astype(float)))
    return np.array(aucs)


def interval(samples):
    return np.percentile(samples, [2.5, 97.5]).round(3).tolist()


def bootstrap_p(samples, null):
    """Two-sided bootstrap p-value, with a +1 correction so that it is never 0."""
    below = (np.sum(samples <= null) + 1) / (len(samples) + 1)
    above = (np.sum(samples >= null) + 1) / (len(samples) + 1)
    return float(min(1.0, 2 * min(below, above)))

Precision and recall need a threshold. It is the value from 0.01 to 0.99 that maximizes the balanced accuracy (the mean
of the recall of copied and of non-copied skills) on the out-of-fold probabilities of the development set.

In [ ]:
def balanced_accuracy(y, p, threshold):
    flagged = p >= threshold
    return float((np.mean(flagged[y == 1]) + np.mean(~flagged[y == 0])) / 2)


def at_threshold(y, p, threshold):
    flagged = p >= threshold
    tp, fp, fn = np.sum(flagged & (y == 1)), np.sum(flagged & (y == 0)), np.sum(~flagged & (y == 1))
    return {"flagged": int(flagged.sum()), "precision": tp / max(tp + fp, 1), "recall": tp / max(tp + fn, 1),
            "balanced_accuracy": balanced_accuracy(y, p, threshold)}

---
## 3.8 Interpretation

The text model is interpreted on about 20,000 holdout skills made of whole authors. Permutation importance shuffles
an attribute, or a whole family at once, and measures how much the AUC drops; its interval comes from resampling
authors.

In [ ]:
def interpretation_sample(hold):
    """Whole authors, in the order of md5(author:seed), until about 20,000 skills."""
    seed, rows = CONFIG["interpretation"]["seed"], CONFIG["interpretation"]["rows"]
    keys = hold["owner"].astype(str).map(lambda owner: hashlib.md5(f"{owner}:{seed}".encode()).hexdigest())
    ordered = hold.assign(_key=keys).sort_values(["_key", "content_hash"])
    per_author = ordered.groupby("_key", sort=False).size()
    keep = per_author.index[(per_author.cumsum() - per_author) < rows]  # authors that start before the limit
    return ordered[ordered["_key"].isin(keep)].drop(columns="_key").reset_index(drop=True)

In [ ]:
def permutation_importance(model, cols, x, y, owners, seed, n_boot, families_only=False):
    """AUC drop when an attribute, or a whole family, is shuffled (mean of 10 shuffles)."""
    rng = np.random.default_rng(seed)
    p_base = model.predict_proba(x[cols])[:, 1]
    targets = {f"family:{name}": columns for name, columns in FAMILIES.items()}
    if not families_only:
        targets.update({c: [c] for c in TEXT})

    # bootstrap weights of each row (the times its author is drawn), the same for every attribute
    authors, author_of_row = np.unique(owners, return_inverse=True)
    boot_rng = np.random.default_rng(seed + 1)
    weights = [np.bincount(boot_rng.integers(0, len(authors), len(authors)),
                           minlength=len(authors))[author_of_row].astype(float) for _ in range(n_boot)]
    auc_base = roc_auc_score(y, p_base)
    auc_base_boot = np.array([roc_auc_score(y, p_base, sample_weight=w) for w in weights])

    rows = []
    for name, group in targets.items():
        group = [c for c in group if c in cols]
        if not group:
            continue
        drops, boot_drops = [], []
        for _ in range(CONFIG["interpretation"]["permutations"]):
            shuffled = x[cols].copy()
            shuffled[group] = shuffled[group].iloc[rng.permutation(len(shuffled))].to_numpy()
            p = model.predict_proba(shuffled)[:, 1]
            drops.append(auc_base - roc_auc_score(y, p))
            boot_drops.append(auc_base_boot - np.array([roc_auc_score(y, p, sample_weight=w) for w in weights]))
        mean_boot = np.mean(boot_drops, axis=0)
        rows.append({"feature": name, "auc_drop": float(np.mean(drops)),
                     "ci95_low": float(np.percentile(mean_boot, 2.5)),
                     "ci95_high": float(np.percentile(mean_boot, 97.5))})
    return pd.DataFrame(rows).sort_values("auc_drop", ascending=False).reset_index(drop=True)

SHAP values split each prediction into the contribution of each attribute, in log-odds. For a logistic
regression, the SHAP value of an attribute is its coefficient times how far the skill is from the average on that
attribute (after the log and the scaling of the pipeline).

---
## 4.1 Exploratory analysis

The dating of the copies outside aggregators (Figure 1c), the time to the first copy and the first appearances by month
(Figures 1a and 1b):

In [ ]:
t0 = pd.to_datetime(dev_all.t0, utc=True)
month = t0.dt.strftime("%Y-%m").where(t0.dt.strftime("%Y-%m") >= "2025-10", "earlier")
per_month = month.value_counts().sort_index(key=lambda m: m.str.replace("earlier", "0"))
status = outside.status.map(STATUS).value_counts(normalize=True)
limit = SNAPSHOT - pd.Timedelta(days=W)

fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))
days = np.arange(91)
ax[0].plot(days, [(lags <= d).mean() for d in days])
ax[0].axvline(W, color="red", ls="--", label=f"W = {W} days")
ax[0].axhline(0.8, color="grey", lw=0.7)
ax[0].set(xlabel="days from first appearance to first copy", ylabel="cumulative share", title="(a) Time to first copy")
ax[0].legend()
ax[1].bar(per_month.index, per_month.values)
x_limit = list(per_month.index).index(limit.strftime("%Y-%m")) - 0.5 + (limit.day - 1) / limit.days_in_month
ax[1].axvline(x_limit, color="red", ls="--", label=f"eligibility limit ({limit.date()})")
ax[1].tick_params(axis="x", rotation=60, labelsize=7)
ax[1].set_title("(b) First appearance (T0) by month")
ax[1].legend()
ax[2].barh(status.index[::-1], status.values[::-1])
ax[2].set(xlim=(0, 1), title="(c) Dating of copies outside aggregators")
plt.tight_layout()
plt.show()
print("first copy, quartiles of days:", lags.quantile([0.25, 0.5, 0.75]).round(1).tolist())

Most counts are zero for many skills and very large for a few (Figure 2a), and many attributes measure almost the
same thing (Figure 2b).

In [ ]:
numeric = [c for c in TEXT if dev[c].nunique() > 2]
skewness = pd.DataFrame({"raw": dev[numeric].astype(float).skew(),
                         "after log1p": np.log1p(dev[numeric].clip(lower=0).astype(float)).skew()})
missing = dev[TEXT + ORIGIN].isna().mean()
print("columns with missing values:", missing[missing > 0].round(4).to_dict())
print("text columns with skewness above 2:", (skewness["raw"] > 2).sum())
display(skewness.sort_values("raw", ascending=False).head(8).round(1))

correlation = dev[TEXT].astype(float).corr("spearman")
fig, ax = plt.subplots(figsize=(7, 6))
image = ax.imshow(correlation.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(TEXT)), TEXT, rotation=90, fontsize=6)
ax.set_yticks(range(len(TEXT)), TEXT, fontsize=6)
fig.colorbar(image, ax=ax, shrink=0.7, label="Spearman correlation")
ax.set_title("(2b) Correlation between text attributes")
plt.show()

The share of copied skills in each quartile of each attribute (Figure 3):

In [ ]:
rates = {}
for column in [c for c in TEXT if dev[c].nunique() > 4]:
    quartile = pd.qcut(dev[column].rank(method="first"), 4, labels=["Q1", "Q2", "Q3", "Q4"])
    rates[column] = dev.groupby(quartile, observed=True).replicated.mean()
invalid = dev.frontmatter_valid == 0
print(f"copied with an invalid front matter: {dev.replicated[invalid].mean():.1%}, "
      f"with a valid one: {dev.replicated[~invalid].mean():.1%}")
pd.DataFrame(rates).T.style.format("{:.1%}").background_gradient(cmap="RdBu_r", axis=None)

---
## 4.2 Choosing a model

Spot-check. Seven algorithms with their default settings, each trained and validated on the 9 partitions
(Figure 4a). This takes a few minutes.

In [ ]:
y_dev = dev.replicated.to_numpy()
rows = []
for repetition, fold, train, valid in folds(dev):
    for algo in CONFIG["models"]["algorithms"]:
        model, cols = pipeline(algo, "text")
        model.fit(dev.iloc[train][cols], y_dev[train])
        p = model.predict_proba(dev.iloc[valid][cols])[:, 1]
        rows.append({"repetition": repetition, "fold": fold, "algorithm": algo,
                     "auc": metrics(y_dev[valid], p)["roc_auc"]})
spot = pd.DataFrame(rows)
spot_summary = spot.groupby("algorithm").auc.agg(["mean", "std"]).sort_values("mean", ascending=False)
finalists = spot_summary.index[:3].tolist()
display(spot_summary.round(3))
print("finalists:", finalists)

Nested search. In each of six outer folds (the first two repetitions), each finalist is measured with its default
settings and with the best configuration of a search that sees only the training part. This is the slowest cell of the
notebook (about ten minutes on Colab).

In [ ]:
rows = []
for repetition, fold, train_rows, valid_rows in folds(dev, repeats=2):
    train, valid = dev.iloc[train_rows], dev.iloc[valid_rows]
    size = {"repetition": repetition, "fold": fold, "train_rows": len(train), "valid_rows": len(valid)}
    for algo in finalists:
        model, cols = pipeline(algo, "text")
        start = time.monotonic()
        model.fit(train[cols], y_dev[train_rows])
        seconds = time.monotonic() - start
        auc = metrics(y_dev[valid_rows], model.predict_proba(valid[cols])[:, 1])["roc_auc"]
        rows.append({**size, "config": f"{algo}|default", "auc": auc, "fit_seconds": seconds})

        tuned = search(algo, train, seed=SEED + repetition - 1)
        auc = metrics(y_dev[valid_rows], tuned.best_estimator_.predict_proba(valid[cols])[:, 1])["roc_auc"]
        rows.append({**size, "config": f"{algo}|tuned", "auc": auc, "fit_seconds": tuned.refit_time_})
    print(f"outer fold {repetition}.{fold} done")
nested = pd.DataFrame(rows)
nested_summary = nested.groupby("config").agg(mean=("auc", "mean"), std=("auc", "std"),
                                              fit_seconds=("fit_seconds", "mean"))
nested_summary.sort_values("mean", ascending=False).round(3)

The choice. The configuration with the highest mean AUC is tested against each of the others:

In [ ]:
tests = compare_to_best(nested)
display(tests.round(3))
tied = [tests.best[0]] + tests.loc[tests.p_holm >= 0.05, "other"].tolist()
print("not significantly worse than the best:", tied)

The rule takes the fastest of the tied configurations. Fit times depend on the machine, so the fastest one here may
not be the fastest one in the study, and a different choice would change every number that follows. The code
therefore uses the choice recorded by the study: logistic regression with the default hyperparameters.

In [ ]:
ALGO, PARAMS = "logistic", {}  # the choice of the study

table4 = spot_summary.rename(columns={"mean": "spot-check", "std": "spot-check std"})
for variant in ("default", "tuned"):
    table4[f"nested, {variant}"] = [nested_summary["mean"].get(f"{a}|{variant}") for a in table4.index]
display(table4.round(3))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
order = spot_summary.index[::-1]
ax[0].boxplot([spot.auc[spot.algorithm == a] for a in order], orientation="horizontal")
ax[0].set_yticks(range(1, len(order) + 1), order)
ax[0].axvline(0.5, color="grey", ls="--", lw=0.8)
ax[0].set(xlabel="AUC (9 folds)", title="(4a) Spot-check, default settings")
order = nested_summary["mean"].sort_values().index
ax[1].boxplot([nested.auc[nested.config == c] for c in order], orientation="horizontal")
ax[1].set_yticks(range(1, len(order) + 1), order)
ax[1].set(xlabel="AUC (6 outer folds)", title="(4b) Nested tuning of the finalists")
plt.tight_layout()
plt.show()

Decision threshold. The text model is validated on the 9 partitions, and its out-of-fold probabilities give the
threshold:

In [ ]:
views = ["text", "size", "origin_pop", "origin_pop+text"]
rows, out_of_fold = [], []
for repetition, fold, train, valid in folds(dev):
    for view in views:
        model, cols = pipeline(ALGO, view, PARAMS)
        model.fit(dev.iloc[train][cols], y_dev[train])
        p = model.predict_proba(dev.iloc[valid][cols])[:, 1]
        rows.append({"view": view, "auc": metrics(y_dev[valid], p)["roc_auc"]})
        if view == "text":
            out_of_fold.append(pd.DataFrame({"y": y_dev[valid], "p": p}))
dev_auc = pd.DataFrame(rows).groupby("view", sort=False).auc.agg(["mean", "std"])
out_of_fold = pd.concat(out_of_fold)

grid = np.round(np.arange(0.01, 1.0, 0.01), 2)
THRESHOLD = float(grid[np.argmax([balanced_accuracy(out_of_fold.y.to_numpy(), out_of_fold.p.to_numpy(), t)
                                  for t in grid])])
print("threshold:", THRESHOLD)
print({k: round(v, 3) for k, v in at_threshold(out_of_fold.y.to_numpy(), out_of_fold.p.to_numpy(), THRESHOLD).items()})
print(f"share of skills flagged: {(out_of_fold.p >= THRESHOLD).mean():.1%}")

---
## 4.3 Does the text predict whether a skill is copied? (RQ1)

Each model is trained on the whole development set and evaluated once on the holdout. The intervals come from 2,000
bootstrap samples of authors (Table 5).

In [ ]:
y_hold, owners = hold.replicated.to_numpy(), hold.owner.astype(str).to_numpy()
final, probs, boot = {}, {}, {}
for view in views:
    model, cols = pipeline(ALGO, view, PARAMS)
    final[view] = (model.fit(dev[cols], y_dev), cols)
    probs[view] = model.predict_proba(hold[cols])[:, 1]
    boot[view] = author_bootstrap(y_hold, probs[view], owners)

table5 = pd.DataFrame({view: metrics(y_hold, p) for view, p in probs.items()}).T
table5.insert(0, "development AUC", [f"{dev_auc['mean'][v]:.3f} ± {dev_auc['std'][v]:.3f}" for v in views])
table5["holdout 95% CI"] = [interval(boot[v]) for v in views]
print(f"holdout: {len(hold):,} skills from {hold.owner.nunique():,} authors, {y_hold.mean():.1%} copied")
table5[["development AUC", "roc_auc", "holdout 95% CI", "pr_auc", "brier"]].round(4)

The two planned tests: H1, the AUC of the text model is above 0.5; and the control, adding the text to the origin
repository raises the AUC. Holm's method adjusts the two p-values together.

In [ ]:
gain = boot["origin_pop+text"] - boot["origin_pop"]
p_h1, p_control = holm([bootstrap_p(boot["text"], 0.5), bootstrap_p(gain, 0.0)])
auc = table5.roc_auc
print(f"H1: text AUC {auc['text']:.3f}, 95% CI {interval(boot['text'])}, Holm-adjusted p = {p_h1:.3f}")
print(f"text minus size only: {auc['text'] - auc['size']:.3f}, 95% CI {interval(boot['text'] - boot['size'])}")
print(f"control: origin + text minus origin: {auc['origin_pop+text'] - auc['origin_pop']:.3f}, "
      f"95% CI {interval(gain)}, Holm-adjusted p = {p_control:.3f}")
print({k: round(v, 3) for k, v in at_threshold(y_hold, probs["text"], THRESHOLD).items()})
print(f"Brier score of predicting {y_hold.mean():.1%} for every skill: {brier_score_loss(y_hold, np.full(len(y_hold), y_hold.mean())):.4f}")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.4))
for view in views:
    fpr, tpr, _ = roc_curve(y_hold, probs[view])
    ax[0].plot(fpr, tpr, label=view)
    precision, recall, _ = precision_recall_curve(y_hold, probs[view])
    keep = recall >= 0.01  # below that, each point rests on a handful of skills
    ax[1].plot(recall[keep], precision[keep], label=view)
ax[0].plot([0, 1], [0, 1], color="grey", ls="--", lw=0.8)
ax[0].set(xlabel="false positive rate", ylabel="true positive rate", title="(5a) ROC curves")
ax[1].axhline(y_hold.mean(), color="grey", ls="--", lw=0.8, label="share copied")
ax[1].set(xlabel="recall", ylabel="precision", ylim=(0, 1), title="(5b) Precision-recall curves")
for a in ax[:2]:
    a.legend(fontsize=7)
matrix = confusion_matrix(y_hold, (probs["text"] >= THRESHOLD).astype(int))
ax[2].imshow(matrix, cmap="Blues")
for (i, j), value in np.ndenumerate(matrix):
    ax[2].text(j, i, f"{value:,}", ha="center", va="center", color="white" if value > matrix.max() / 2 else "black")
ax[2].set_xticks([0, 1], ["predicted\nnot copied", "predicted\ncopied"])
ax[2].set_yticks([0, 1], ["not\ncopied", "copied"])
ax[2].set_title(f"(5c) Text model, threshold {THRESHOLD:.2f}")
plt.tight_layout()
plt.show()

---
## 4.4 How robust is the result?

Each analysis changes one choice, retrains the text model on the development set and evaluates it on the holdout, with
500 bootstrap samples (Table 6). The aggregator variants recompute T0 and the time to first copy with other limits.

In [ ]:
def robustness(train, test):
    model, cols = pipeline(ALGO, "text", PARAMS)
    model.fit(train[cols], train["replicated"])
    y, p = test["replicated"].to_numpy(), model.predict_proba(test[cols])[:, 1]
    samples = author_bootstrap(y, p, test["owner"].astype(str).to_numpy(), n_boot=CONFIG["evaluation"]["n_boot_robustness"])
    return {"skills": len(test), "copied": round(y.mean(), 3), "auc": round(roc_auc_score(y, p), 3), "95% CI": interval(samples)}


analyses = {}
for w in (30, 90):
    analyses[f"window of {w} days"] = robustness(eligible("dev", w), eligible("holdout", w))

variants = table.copy()
for suffix, limit in (("_agg500", 500), ("_aggnone", None)):
    timing = first_appearance(copies, limit)[["content_hash", "t0", "first_copy_lag_days"]]
    variants = variants.merge(timing.add_suffix(suffix).rename(columns={f"content_hash{suffix}": "content_hash"}),
                              on="content_hash", how="left")
analyses["aggregators: limit of 500 skills"] = robustness(eligible("dev", W, variants, "_agg500"),
                                                          eligible("holdout", W, variants, "_agg500"))
analyses["aggregators: no exclusion"] = robustness(eligible("dev", W, variants, "_aggnone"),
                                                   eligible("holdout", W, variants, "_aggnone"))

cutoff = dev.t0_used.quantile(0.70)  # learn from the earlier 70% of the development skills
analyses[f"temporal split (T0 after {cutoff.date()})"] = robustness(dev[dev.t0_used < cutoff], hold[hold.t0_used >= cutoff])
pd.DataFrame(analyses).T

---
## 4.5 What the model learned (RQ2)

Permutation importance of the families and of the attributes, on the interpretation sample (Figure 6). This takes a
few minutes.

In [ ]:
model, cols = final["text"]
sample = interpretation_sample(hold)
importance = permutation_importance(model, cols, sample, sample.replicated.to_numpy(), sample.owner.astype(str).to_numpy(),
                                    SEED, CONFIG["interpretation"]["n_boot"])
print(f"{len(sample):,} skills from {sample.owner.nunique():,} authors")
families = importance[importance.feature.str.startswith("family:")]
attributes_importance = importance[~importance.feature.str.startswith("family:")]
display(families.round(3))
display(attributes_importance.head(10).round(3))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for a, rows, title in ((ax[0], families, "(6a) Attribute families"),
                       (ax[1], attributes_importance.head(10), "(6b) Ten most important attributes")):
    rows = rows.iloc[::-1]
    a.barh(rows.feature.str.removeprefix("family:"), rows.auc_drop,
           xerr=[(rows.auc_drop - rows.ci95_low).clip(lower=0), (rows.ci95_high - rows.auc_drop).clip(lower=0)])
    a.set(xlabel="AUC drop when shuffled (95% CI)", title=title)
plt.tight_layout()
plt.show()

Is the order of the families stable? The same permutation on the validation part of each of the 9 development folds:

In [ ]:
by_fold = {}
for repetition, fold, train, valid in folds(dev):
    fold_model, fold_cols = pipeline(ALGO, "text", PARAMS)
    fold_model.fit(dev.iloc[train][fold_cols], y_dev[train])
    part = dev.iloc[valid].reset_index(drop=True)
    drops = permutation_importance(fold_model, fold_cols, part, y_dev[valid], part.owner.astype(str).to_numpy(),
                                   SEED + 10 * repetition + fold, n_boot=1, families_only=True)
    by_fold[f"{repetition}.{fold}"] = drops.set_index("feature").auc_drop
stability = pd.DataFrame(by_fold)
stability["holdout"] = families.set_index("feature").auc_drop
display(stability.rank(ascending=False).astype(int))
print("Spearman correlation with the holdout order:",
      stability.rank().corr("spearman")["holdout"].drop("holdout").round(2).tolist())

SHAP values give the direction and the shape of each effect (Figure 7).

In [ ]:
import shap

x_model = model[:-1].transform(sample[cols])  # the data as the model sees it
explainer = shap.LinearExplainer(model.named_steps["model"], shap.maskers.Independent(x_model, max_samples=len(x_model)))
shap_values = explainer.shap_values(x_model)
names = list(model.named_steps["pre"].get_feature_names_out())
position = {c: next(k for k, n in enumerate(names) if n.endswith(f"__{c}")) for c in cols}
mean_abs = pd.Series({c: np.abs(shap_values[:, k]).mean() for c, k in position.items()}).sort_values(ascending=False)
display(mean_abs.head(12).round(3).to_frame("mean |SHAP|"))

top12 = mean_abs.index[:12][::-1]
rng = np.random.default_rng(0)
pick = rng.choice(len(sample), size=5000, replace=False)
fig, ax = plt.subplots(figsize=(10, 5))
for row, c in enumerate(top12):
    color = pd.Series(sample[c].to_numpy(dtype=float)[pick]).rank(pct=True)
    ax.scatter(shap_values[pick, position[c]], row + rng.uniform(-0.3, 0.3, len(pick)), c=color, cmap="coolwarm", s=2, alpha=0.5)
ax.set_yticks(range(len(top12)), top12, fontsize=8)
ax.axvline(0, color="grey", lw=0.7)
ax.set(xlabel="SHAP value (log-odds); color from low (blue) to high (red) value", title="(7a) SHAP values")
plt.show()

fig, ax = plt.subplots(1, 5, figsize=(16, 3.2))
for a, c in zip(ax, attributes_importance.feature.head(5)):
    x = sample[c].to_numpy(dtype=float)
    low, high = np.nanpercentile(x, [1, 99])
    inside = (x >= low) & (x <= high)
    a.scatter(x[inside], shap_values[inside, position[c]], s=2, alpha=0.3)
    a.axhline(0, color="grey", lw=0.7)
    a.set_title(c, fontsize=9)
ax[0].set_ylabel("SHAP value (log-odds)")
plt.tight_layout()
plt.show()

---
## Appendix A — Reproducibility

This notebook starts from the outputs of the collection. The collection itself is in the code of the
[repository](https://github.com/podorize/skillreuse): the verified download of GitSkills, the sample, the dating of
the copies through the GitHub API, and the text attributes and repository metadata of the sampled skills. The README
explains how to run it, from checking the published results to redoing everything from scratch.